# Film Reel-ational Data — Prototyping & Schema Mapping Notebook
**Authors:** Andrew & Isaac  
**Milestones Covered:**
- Data ingestion (CSV / JSON column parsing)
- Mapping Local Schemas (Kaggle CSVs) to Global Relational Schema (SQLite)
- Conversion to SQLite database (`movies.db`)
- Verification queries for Multi-Table Search, CRUD, **Bombed at the Box Office**, and **Hidden Gems**

## 1. Local Schema to Global Schema Mapping
| Global SQLite Table | Global Attributes | Local Source (Kaggle CSVs) | Transformation / Constraints |
|---|---|---|---|
| `Movies` | `movie_id (PK)`, `title`, `budget`, `revenue`, `vote_average`, `vote_count` | `movies_metadata.csv` / `tmdb_5000_movies.csv` (`id`, `title`, `budget`, `revenue`, `vote_average`, `vote_count`) | `CHECK (vote_average BETWEEN 0 AND 10)` |
| `Actors` | `actor_id (PK)`, `actor_name` | `credits.csv` / `tmdb_5000_credits.csv` (`cast` JSON array -> `id`, `name`) | Deduplicated on `actor_id` |
| `Movie_Cast` | `movie_id (FK)`, `actor_id (FK)`, `character_name`, `PK(movie_id, actor_id)` | `credits.csv` (`id` + `cast` JSON array -> `id`, `character`) | `ON DELETE CASCADE` |
| `Genre_Types` | `genre_id (PK)`, `genre_name` | `movies_metadata.csv` (`genres` JSON array -> `id`, `name`) | 3NF Lookup Table |
| `Genres` | `movie_id (FK)`, `genre_id (FK)`, `PK(movie_id, genre_id)` | `movies_metadata.csv` (`id` + `genres` JSON array -> `id`) | `ON DELETE CASCADE` |

In [ ]:
import sqlite3
from ingest_data import run_ingestion
import database as db

# Initialize and ingest data (uses ./data CSVs if present, or seeds real TMDB sample dataset)
counts = run_ingestion()
print('Database Summary:', counts)

In [ ]:
# Multi-Table Search across Movies, Actors, Movie_Cast, and Genres
results, sql = db.search_movies(actor_query='Cillian Murphy')
for r in results:
    print(f"{r['title']} | Rating: {r['vote_average']} | Genres: {r['genres']} | Cast: {r['cast_names']}")

In [ ]:
# Feature 1: 'Bombed at the Box Office' (High budget, revenue < budget)
bombs, _ = db.get_box_office_bombs(min_budget=50_000_000, limit=5)
print('Top 5 Box Office Bombs:')
for b in bombs:
    print(f"- {b['title']}: Budget=${b['budget']:,}, Revenue=${b['revenue']:,}, Net Loss=-${b['net_loss']:,} ({b['roi_percent']}% ROI)")

In [ ]:
# Feature 2: 'Hidden Gems' (High vote_average, low/moderate vote_count & budget)
gems, _ = db.get_hidden_gems(min_rating=7.5, min_votes=50, max_votes=4500, limit=5)
print('Top 5 Hidden Gems:')
for g in gems:
    print(f"- {g['title']}: Rating={g['vote_average']} ({g['vote_count']} votes), Gem Score={g['gem_score']}, Genres={g['genres']}")